<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-17</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>INT4 양자화(INT4 Quantization)</strong>에 대해 학습합니다.</br></br>
>BitsAndBytesConfig로 INT4 양자화를 적용하고 FP16과 성능을 비교한 뒤, 환경 변화 입력에서의 취약성을 체감해봅시다.

</br>

# INT4 양자화 (4-bit Quantization)
> 가중치를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">4비트 정수</mark>로 표현하여 모델 크기를 1/8로 압축합니다.

INT4는 FP16보다 훨씬 극단적인 압축입니다. FP16은 16비트로 숫자를 표현하지만, INT4는 단 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">4비트</mark>만 사용하여 고작 **2⁴ = 16가지** 값만 표현할 수 있습니다. 7B 모델 기준으로 FP16은 14GB가 필요하지만, INT4는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">3.5GB</mark>(FP16 대비 1/4)까지 줄어듭니다. 값의 범위가 줄어들면서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">약간의 정밀도 손실</mark>이 발생하지만, NF4(NormalFloat4)처럼 가중치 분포에 최적화된 양자화 방식을 쓰면 실제 품질 저하는 생각보다 작습니다. 결국 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">대폭적인 메모리 절감을 아주 작은 정밀도 손실로 교환</mark>하는 것이 INT4 양자화의 핵심 가치입니다.

이 내용은 FP16 양자화의 원리(Ch.5-2_001 참고), 모델 가중치, NF4(정규분포 가정 하의 4비트 양자화), QLoRA(INT4 모델 위에 LoRA를 적용하는 초저메모리 파인튜닝 기법) 등의 개념을 바탕으로 합니다.

</br>

## BitsAndBytesConfig
> Hugging Face의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">bitsandbytes 라이브러리</mark>로 4비트 양자화를 설정합니다.

In [ ]:
# TODO 1: 양자화 설정을 구성하여 4비트 양자화 활성화, NF4 양자화 타입, FP16 연산 정밀도, 이중 양자화를 설정하고, 해당 설정으로 모델을 로드해봅시다.

from transformers import BitsAndBytesConfig, AutoModelForCausalLM
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4비트 양자화 활성화
    bnb_4bit_quant_type="nf4",            # NF4 양자화 타입
    bnb_4bit_compute_dtype=torch.float16, # 연산 시 FP16 사용
    bnb_4bit_use_double_quant=True,       # 이중 양자화
)

model = AutoModelForCausalLM.from_pretrained(
    "model_name",
    quantization_config=quantization_config,
    device_map="auto"
)

# 메모리 확인
mem_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU 메모리 사용량: {mem_gb:.2f} GB")
print(f"양자화 설정: {quantization_config.to_dict()}")

</br>

## 주요 설정 파라미터

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th style="text-align:center">권장값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>load_in_4bit</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">4비트 양자화</mark> 활성화</td><td style="text-align:center">True</td></tr>
    <tr><td style="text-align:center"><code>bnb_4bit_quant_type</code></td><td>양자화 방식</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">"nf4"</mark></td></tr>
    <tr><td style="text-align:center"><code>bnb_4bit_compute_dtype</code></td><td>연산 시 정밀도</td><td style="text-align:center">torch.float16</td></tr>
    <tr><td style="text-align:center"><code>bnb_4bit_use_double_quant</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이중 양자화</mark> (추가 압축)</td><td style="text-align:center">True</td></tr>
  </tbody>
</table>

</br>

## NF4 (NormalFloat4)
> 가중치가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정규분포를 따른다는 가정</mark>하에 최적화된 4비트 양자화 방식입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">양자화 타입</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">FP4</td><td>일반 4비트 부동소수점</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">NF4</mark></td><td>정규분포 최적화, 더 높은 정밀도</td></tr>
  </tbody>
</table>

</br>

## FP16 vs INT4 비교

In [ ]:
# TODO 2: 7B 모델 기준 FP16(14.0GB, 100%), INT4 NF4(3.5GB, 25%), INT4+DQ(3.2GB, 23%)의 메모리 비교 표를 출력해봅시다.

print(f"{'모드':<12} {'메모리':>8} {'비율':>6}")
print(f"{'FP16':<12} {'14.0 GB':>8} {'100%':>6}")
print(f"{'INT4 (NF4)':<12} {'3.5 GB':>8} {'25%':>6}")
print(f"{'INT4+DQ':<12} {'3.2 GB':>8} {'23%':>6}")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">모드</th>
      <th style="text-align:center">메모리</th>
      <th style="text-align:center">비율</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">FP16</td><td style="text-align:center">14.0 GB</td><td style="text-align:center">100%</td></tr>
    <tr><td style="text-align:center">INT4 (NF4)</td><td style="text-align:center">3.5 GB</td><td style="text-align:center">25%</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">INT4+DQ</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">3.2 GB</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">23%</mark></td></tr>
  </tbody>
</table>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교 항목</th>
      <th style="text-align:center">FP16</th>
      <th style="text-align:center">INT4 (NF4)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">메모리</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">100%</mark> (기준)</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">약 25%</mark></td></tr>
    <tr><td style="text-align:center">추론 속도</td><td style="text-align:center">빠름</td><td style="text-align:center">비슷하거나 약간 느림</td></tr>
    <tr><td style="text-align:center">품질</td><td style="text-align:center">원본과 동일</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">약간의 정밀도 손실</mark></td></tr>
    <tr><td style="text-align:center">GPU 요구사항</td><td style="text-align:center">높음</td><td style="text-align:center">낮음</td></tr>
  </tbody>
</table>

</br>

# QLoRA (Quantized LoRA)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">INT4 양자화 + LoRA</mark>를 결합하여 극소량의 메모리로 파인튜닝합니다.

In [ ]:
# TODO 3: LoRA 설정을 구성하고, PEFT 모델을 생성하여 INT4 모델에 LoRA를 적용한 뒤 학습 가능한 파라미터를 확인해봅시다.

from peft import get_peft_model, LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)  # INT4 모델 + LoRA
model.print_trainable_parameters()

💡QLoRA의 핵심 아이디어
> 기본 모델은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">INT4로 동결</mark>하고, LoRA 어댑터만 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">FP16으로 학습</mark>합니다.</br>
> 7B 모델도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단일 GPU(16GB)</mark>에서 파인튜닝이 가능합니다.

💡양자화 모델의 주의점
> INT4 모델은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">노이즈에 더 민감</mark>합니다.</br>
> 프롬프트가 불명확하면 FP16보다 품질 저하가 더 클 수 있으므로, 명확하고 구조화된 프롬프트를 사용하세요.

</br>

# 환경 변화 입력 테스트 (Distribution Shift Test)
> 양자화 모델이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">오타, 노이즈, 모호한 입력</mark>에 얼마나 취약한지 직접 체감합니다.

실제 서비스 환경에서 사용자 입력은 학습 데이터처럼 깔끔하지 않습니다. 오타가 섞이거나, 불필요한 표현이 끼어들거나, 맥락이 부족한 모호한 질문이 들어옵니다. FP16 모델은 이런 입력도 비교적 잘 처리하지만, INT4 양자화 모델은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정밀도 손실이 누적</mark>되어 경계 케이스에서 품질이 크게 떨어집니다. 이 현상을 직접 관찰하여 양자화의 트레이드오프를 체감해봅시다.

## 환경 변화 유형

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">유형</th>
      <th>예시</th>
      <th style="text-align:center">특징</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">정상</td><td>"서울에서 부산까지 KTX로 얼마나 걸리나요?"</td><td style="text-align:center">명확한 질문</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">오타</mark></td><td>"서울에셔 부산까지 KTX로 얼마나 걸리나요?"</td><td style="text-align:center">오타 포함</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">노이즈</mark></td><td>"서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?"</td><td style="text-align:center">불필요한 표현</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모호함</mark></td><td>"그거 얼마나 걸려?"</td><td style="text-align:center">맥락 부족</td></tr>
    <tr><td style="text-align:center">조건 변화</td><td>"밤 11시에 출발하면 다음날 아침에 도착할 수 있나요?"</td><td style="text-align:center">복잡한 조건</td></tr>
  </tbody>
</table>

</br>

## 환경 변화 입력 샘플 정의

In [ ]:
# TODO 4: 정상/오타/노이즈/모호함/조건변화 5가지 유형의 환경 변화 입력 샘플을 딕셔너리 리스트로 정의해봅시다.

test_samples = [
    {
        "type": "정상",
        "prompt": "서울에서 부산까지 KTX로 얼마나 걸리나요?",
        "expected_quality": "높음"
    },
    {
        "type": "오타",
        "prompt": "서울에셔 부산까지 KTX로 얼마나 걸리나요?",
        "expected_quality": "중간"
    },
    {
        "type": "노이즈",
        "prompt": "서울에서 부산까지... 음... KTX로 얼마나 걸리나요? 대략?",
        "expected_quality": "중간"
    },
    {
        "type": "모호함",
        "prompt": "그거 얼마나 걸려?",
        "expected_quality": "낮음"
    },
    {
        "type": "조건변화",
        "prompt": "밤 11시에 출발하면 다음날 아침에 도착할 수 있나요?",
        "expected_quality": "낮음"
    },
]

print(f"환경 변화 입력 샘플: {len(test_samples)}개")
for sample in test_samples:
    print(f"  [{sample['type']}] {sample['prompt']}")

</br>

## INT4 모델에서 환경 변화 입력 테스트

In [ ]:
# TODO 5: 각 환경 변화 입력 샘플을 INT4 양자화 모델에 입력하여 추론하고, 입력 유형별 응답을 출력해봅시다.

env_change_results = []

print("=" * 70)
print("[INT4 모델 - 환경 변화 입력 테스트]")
print("=" * 70)

for sample in test_samples:
    inputs = tokenizer(sample["prompt"], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    env_change_results.append({
        "type": sample["type"],
        "prompt": sample["prompt"],
        "response": response,
        "expected_quality": sample["expected_quality"]
    })

    print(f"\n[{sample['type']}] (예상 품질: {sample['expected_quality']})")
    print(f"  입력: {sample['prompt']}")
    print(f"  응답: {response[:200]}")
    print("-" * 70)

In [ ]:
# TODO 6: 환경 변화 입력 테스트 결과를 요약하여 입력 유형별 품질 저하 현상을 출력해봅시다.

print("=" * 50)
print("[환경 변화 입력 품질 저하 관찰]")
print("=" * 50)
print("- 오타: 오타를 이해하지 못하거나 이상한 응답 생성")
print("- 노이즈: 불필요한 표현에 혼란, 핵심을 놓침")
print("- 모호함: 맥락 부족으로 엉뚱한 응답 생성")
print("- 조건변화: 복잡한 조건 처리 실패")
print()
print("→ 양자화로 인한 정밀도 손실이 비정상 입력에서 품질 저하로 이어집니다.")
print("→ 이 문제를 해결하기 위해 Ch.5-2_003에서 TTP 전략을 학습합니다.")

## 양자화 모델이 환경 변화에 취약한 이유

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">원인</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정밀도 손실 누적</mark></td><td>4비트로 표현할 수 있는 값이 16개뿐이라 미세한 가중치 차이가 사라지고, 이 오차가 레이어를 거치며 누적됩니다.</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Attention 변화</mark></td><td>양자화된 가중치로 계산한 attention score에 미세한 차이가 생겨 다른 토큰에 집중하게 됩니다.</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">경계 케이스 취약</mark></td><td>학습 데이터에서 드문 패턴(오타, 노이즈)은 가중치가 약하여 양자화에 민감합니다.</td></tr>
  </tbody>
</table>

💡환경 변화와 양자화의 관계
> 양자화된 모델은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">"일반적인 입력"에서는 FP16과 거의 동일한 품질</mark>을 보이지만, "비정상적인 입력(오타, 노이즈, 모호함)"에서는 품질이 크게 저하됩니다.</br>
> 이를 보완하기 위해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">TTP(Test-Time Prompting)</mark> 전략을 사용합니다. (Ch.5-2_003 참고)